<a href="https://colab.research.google.com/github/mistervio/1.2.-Site-For-Import/blob/main/Final_project_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Установка pandas
print("Установка pandas...")
!pip install pandas

# Проверка установки
import pandas as pd
print(f"\nPandas установлен, версия: {pd.__version__}")

Установка pandas...

Pandas установлен, версия: 2.2.3


In [ ]:
# Создание структуры проекта
import os
from pathlib import Path

# Создаем все необходимые папки
folders = ['data', 'reports', 'logs', 'src']
for folder in folders:
    Path(folder).mkdir(exist_ok=True)
    print(f"Создана папка: {folder}/")

# Проверяем созданную структуру
print("\nСтруктура проекта:")
for folder in folders:
    if Path(folder).exists():
        print(f"{folder}/")

Создана папка: data/
Создана папка: reports/
Создана папка: logs/
Создана папка: src/

Структура проекта:
data/
reports/
logs/
src/


In [ ]:
# Создание файла requirements.txt
with open("requirements.txt", "w", encoding='utf-8') as f:
    f.write("pandas\n")

print("Создан файл: requirements.txt")

# Проверяем содержимое
with open("requirements.txt", "r") as f:
    content = f.read()
    print(f"\nСодержимое requirements.txt:\n{content}")

Создан файл: requirements.txt

Содержимое requirements.txt:
pandas



In [ ]:
# Создание файла config.py
config_code = '''"""
Конфигурационный файл для проекта анализа заказов.
"""

from pathlib import Path

# Пути к папкам
DATA_DIR = Path("data")
REPORTS_DIR = Path("reports")
LOGS_DIR = Path("logs")

# Настройки для фильтрации
STATUS_COLUMN = "status"
DELIVERED_STATUS = "Delivered"

# Настройки для вывода
RESULTS_FILE = "analysis_results.csv"
'''

with open("config.py", "w", encoding='utf-8') as f:
    f.write(config_code)

print("Создан файл: config.py")

# Проверяем содержимое
with open("config.py", "r") as f:
    print(f"\nСодержимое config.py:\n{f.read()}")

Создан файл: config.py

Содержимое config.py:
"""
Конфигурационный файл для проекта анализа заказов.
"""

from pathlib import Path

# Пути к папкам
DATA_DIR = Path("data")
REPORTS_DIR = Path("reports")
LOGS_DIR = Path("logs")

# Настройки для фильтрации
STATUS_COLUMN = "status"
DELIVERED_STATUS = "Delivered"

# Настройки для вывода
RESULTS_FILE = "analysis_results.csv"



In [ ]:
# Создание файла src/analyzer.py
analyzer_code = '''
import pandas as pd
import logging
from pathlib import Path
from datetime import datetime


class OrderAnalyzer:
    def __init__(self, data_dir, reports_dir, logs_dir, status_column, delivered_status, results_file):
        self.data_dir = Path(data_dir)
        self.reports_dir = Path(reports_dir)
        self.logs_dir = Path(logs_dir)
        self.status_column = status_column
        self.delivered_status = delivered_status
        self.results_file = results_file
        self.processed_files = 0
        self.error_files = 0
        self.results = []


        # Настройка логов
        log_file = self.logs_dir / f"processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[logging.FileHandler(log_file, encoding='utf-8'), logging.StreamHandler()]
        )
        self.logger = logging.getLogger(__name__)

    def find_csv_files(self):
        return list(self.data_dir.glob("*.csv"))

    def process_file(self, file_path):
        try:
            if file_path.stat().st_size == 0:
                self.logger.error(f"Файл {file_path.name} пуст")
                return None

            # Чтение файла
            df = pd.read_csv(file_path)

            # Проверка наличия колонок
            if 'status' not in df.columns or 'total_amount' not in df.columns:
                self.logger.error(f"В файле {file_path.name} нет нужных колонок")
                return None

            # Фильтрация доставленных
            delivered = df[df['status'] == 'Delivered']

            # Расчет метрик
            metrics = {
                'file_name': file_path.name,
                'total_revenue': round(delivered['total_amount'].sum(), 2) if not delivered.empty else 0,
                'average_order_value': round(delivered['total_amount'].mean(), 2) if not delivered.empty else 0,
                'order_count': len(delivered)
            }

            self.logger.info(f"Файл {file_path.name} обработан: {metrics['order_count']} заказов")
            return metrics

        except Exception as e:
            self.logger.error(f"Ошибка в файле {file_path.name}: {e}")
            return None

    def run(self):
        """Запускает обработку всех файлов"""
        files = self.find_csv_files()

        if not files:
            print("В папке data/ нет CSV-файлов")
            return

        print(f"Найдено файлов: {len(files)}")
        print("Начинаю обработку...")

        for file_path in files:
            result = self.process_file(file_path)
            if result:
                self.results.append(result)
                self.processed_files += 1
                print(f"{file_path.name} - обработан")
            else:
                self.error_files += 1
                print(f"{file_path.name} - пропущен")

        # Вывод статистики
        print(f"\\nОбработано: {self.processed_files}, Ошибок: {self.error_files}")

        # Сохранение результатов
        if self.results:
            df_results = pd.DataFrame(self.results)
            df_results.to_csv(self.reports_dir / self.results_file, index=False)
            print(f"Результаты сохранены в {self.reports_dir / self.results_file}")
'''

with open("src/analyzer.py", "w", encoding='utf-8') as f:
    f.write(analyzer_code)

print("Создан файл: src/analyzer.py")

Создан файл: src/analyzer.py


In [ ]:
# Создание файла run.py
run_code = '''#!/usr/bin/env python
"""
Главный скрипт для запуска анализа заказов.
"""

import sys
from pathlib import Path

# Добавляем src в путь для импорта
sys.path.insert(0, str(Path(__file__).parent / "src"))

from analyzer import OrderAnalyzer
from config import (
    DATA_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    STATUS_COLUMN,
    DELIVERED_STATUS,
    RESULTS_FILE
)


def main():
    """Основная функция запуска анализа."""
    print("="*50)
    print("АНАЛИЗ ДАННЫХ О ЗАКАЗАХ")
    print("="*50)

    try:
        analyzer = OrderAnalyzer(
            data_dir=DATA_DIR,
            reports_dir=REPORTS_DIR,
            logs_dir=LOGS_DIR,
            status_column=STATUS_COLUMN,
            delivered_status=DELIVERED_STATUS,
            results_file=RESULTS_FILE
        )

        analyzer.run()

    except Exception as e:
        print(f"\\nОшибка: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()
'''

with open("run.py", "w", encoding='utf-8') as f:
    f.write(run_code)

print("Создан файл: run.py")

# Проверяем содержимое
with open("run.py", "r") as f:
    print(f"\nСодержимое run.py:\n{f.read()}")

Создан файл: run.py

Содержимое run.py:
#!/usr/bin/env python
"""
Главный скрипт для запуска анализа заказов.
"""

import sys
from pathlib import Path

# Добавляем src в путь для импорта
sys.path.insert(0, str(Path(__file__).parent / "src"))

from analyzer import OrderAnalyzer
from config import (
    DATA_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    STATUS_COLUMN,
    DELIVERED_STATUS,
    RESULTS_FILE
)


def main():
    """Основная функция запуска анализа."""
    print("="*50)
    print("АНАЛИЗ ДАННЫХ О ЗАКАЗАХ")
    print("="*50)
    
    try:
        analyzer = OrderAnalyzer(
            data_dir=DATA_DIR,
            reports_dir=REPORTS_DIR,
            logs_dir=LOGS_DIR,
            status_column=STATUS_COLUMN,
            delivered_status=DELIVERED_STATUS,
            results_file=RESULTS_FILE
        )
        
        analyzer.run()
        
    except Exception as e:
        print(f"\nОшибка: {e}")
        sys.exit(1)


if __name__ == "__main__":
    main()



In [ ]:
!python run.py

АНАЛИЗ ДАННЫХ О ЗАКАЗАХ
Найдено файлов: 3
Начинаю обработку...
2026-09-05 20:54:39,910 - INFO - Файл order_100000.csv обработан: 14287 заказов
order_100000.csv - обработан
2026-09-05 20:54:39,926 - INFO - Файл order_10000.csv обработан: 1417 заказов
order_10000.csv - обработан
2026-09-05 20:54:39,941 - ERROR - В файле order_bag.csv нет нужных колонок
order_bag.csv - пропущен

Обработано: 2, Ошибок: 1
Результаты сохранены в reports/analysis_results.csv
